In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as f

In [2]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("OlympicsDevelopment")
    .getOrCreate()
)

spark

In [3]:
DATA_ROOT = Path("../data/test_output")

#### Imports and loaded tables

In [4]:

bronze_athletes = spark.read.parquet(f'{DATA_ROOT}/bronze/athlete_events')
bronze_noc = spark.read.parquet(f'{DATA_ROOT}/bronze/noc_regions')
dim_athlete = spark.read.parquet(f'{DATA_ROOT}/gold/dim_athlete')
dim_games = spark.read.parquet(f'{DATA_ROOT}/gold/dim_games')
dim_event = spark.read.parquet(f'{DATA_ROOT}/gold/dim_event')
dim_noc = spark.read.parquet(f'{DATA_ROOT}/gold/dim_noc')
fact_participation = spark.read.parquet(f'{DATA_ROOT}/gold/fact_participation')


##### Participation and medal trends by year

In [5]:
yearly_analytics = (
    fact_participation
    .groupBy('year')
    .agg(
        f.sum('participation_count').alias('participations'),
        f.sum('medal_count').alias('medals'),
        f.sum('gold_count').alias('gold_medals'),
        f.sum('silver_count').alias('silver_medals'),
        f.sum('bronze_count').alias('bronze_medals')
    )
    .withColumn(
        'medal_rate',
        f.round(
            f.col('medals') / f.col('participations'), 4
        )
    )
    .orderBy(
        f.col('year')
    )
)

yearly_analytics.show(100, truncate=False)

+----+--------------+------+-----------+-------------+-------------+----------+
|year|participations|medals|gold_medals|silver_medals|bronze_medals|medal_rate|
+----+--------------+------+-----------+-------------+-------------+----------+
|2000|1767          |307   |85         |101          |121          |0.1737    |
|2002|1221          |180   |67         |57           |56           |0.1474    |
|2004|1795          |307   |104        |118          |85           |0.171     |
|2006|1248          |183   |86         |42           |55           |0.1466    |
|2008|1803          |346   |124        |113          |109          |0.1919    |
|2010|1279          |189   |71         |65           |53           |0.1478    |
|2012|1814          |352   |110        |137          |105          |0.194     |
|2014|1373          |213   |68         |88           |57           |0.1551    |
|2016|1818          |325   |109        |102          |114          |0.1788    |
+----+--------------+------+-----------+

##### Medal totals by NOC

In [7]:
current_dim_noc = (
    dim_noc
    .filter(
        f.col("is_current")
    )
    .select(
        "noc_key",
        "noc_code",
        "region",
    )
)

medals_by_noc = (
    fact_participation
    .join(
        current_dim_noc,
        on="noc_key",
        how="inner",
    )
    .groupBy(
        "noc_code",
        "region"
    )
    .agg(
        f.sum("participation_count").alias("participations"),
        f.sum("gold_count").alias("gold"),
        f.sum("silver_count").alias("silver"),
        f.sum("bronze_count").alias("bronze"),
        f.sum("medal_count").alias("total_medals")
    )
    .withColumn(
        "medal_rate",
        f.round(
            f.col("total_medals") / f.col("participations"), 4),
    )
    .orderBy(
        f.desc("total_medals")
    )
)

medals_by_noc.show(30, truncate=False)

+--------+---------------------------+--------------+----+------+------+------------+----------+
|noc_code|region                     |participations|gold|silver|bronze|total_medals|medal_rate|
+--------+---------------------------+--------------+----+------+------+------------+----------+
|USA     |United States of America   |1080          |114 |122   |90    |326         |0.3019    |
|GER     |Federal Republic of Germany|1080          |90  |72    |74    |236         |0.2185    |
|NED     |Netherlands                |859           |71  |83    |63    |217         |0.2526    |
|CAN     |Canada                     |1080          |101 |45    |59    |205         |0.1898    |
|SWE     |Sweden                     |1080          |53  |85    |54    |192         |0.1778    |
|CHN     |China                      |1057          |84  |51    |56    |191         |0.1807    |
|KOR     |South Korea                |976           |80  |48    |49    |177         |0.1814    |
|AUS     |Australia           

In [8]:
spark.stop()